In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("data/Churn_Modelling.csv")

In [3]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data.isna().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [6]:
data.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [ ]:
#preprocessing the data
# Drop irrelevant data columns
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis = 1)
data.head()

In [ ]:
label = LabelEncoder()
data["Gender"] = label.fit_transform(data["Gender"])
data

In [ ]:
# Apply Onehot encoding on column - Geography --> this will create an array over here
onehot_encoder_geo = OneHotEncoder()
geo_encoder = onehot_encoder_geo.fit_transform(data[["Geography"]]).toarray()
geo_encoder

In [ ]:
# here we transform that array into dataframe with get_feature_names_out of Geography column
v = pd.DataFrame(geo_encoder, columns = onehot_encoder_geo.get_feature_names_out(["Geography"]))
v

In [ ]:
# then we will concat the data and v dataframes where created the onehot encoding.
data = pd.concat([data.drop("Geography", axis=1), v], axis=1)
data

In [ ]:
# Now we are creating pickle file for label encoder and one hot encoder separately
# 'with open' will open and close the file at a time.
# we are creating pickle file thats why we used pickle.dump which will create pickle file
with open("label.pkl", "wb") as file:
    pickle.dump(label,file)

In [ ]:
with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(onehot_encoder_geo,file)

In [ ]:
X = data.drop("Exited", axis=1)
y = data["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train.shape

In [ ]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
i = X_train.shape[1]

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential  # for sequential model
from tensorflow.keras.layers import Dense # for dense layers, neurons
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [ ]:
#Build our ANN model
model = Sequential(
    [
        Dense(64, activation="relu", input_shape=(i,)), # Hidden layer connected with input layer 
        Dense(32,activation="relu"),    # hidden layer 2
        Dense(1, activation="sigmoid")  # output layer
    ]
)

In [ ]:
model.summary()

In [ ]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate= 0.01) # 0 and 1
loss = tensorflow.keras.losses.BinaryCrossentropy()  #binary classification

print(loss)

In [ ]:
#3 compile the model
# metrics ---> Classification accuracy, f1 score, precision, recall
# regression ---> MSE, MAE

model.compile(optimizer= opt, loss = "binary_crossentropy", metrics= ["accuracy"])

In [ ]:
# set up tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [ ]:
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

In [ ]:
model.save("model.h5")  # saves neural network, another file extension is .keras

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit/20260803-154610